# 04_LULC_AI
## Goals
- Demonstrate rule-based LULC using NDVI/NDWI
- Simple ML inference (RandomForest) on spectral indices


In [ ]:
import numpy as np
import xarray as xr
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

print('ML stack ready')


## Prepare spectral indices (or use dummy data)
If previous datasets exist use them; else we create synthetic arrays to illustrate workflow.


In [ ]:
# Create synthetic NDVI, NDWI arrays for illustration
try:
    ndvi_mean
    ndwi_mean
except NameError:
    ndvi_mean = xr.DataArray(np.random.rand(200,200), dims=('y','x'))
    ndwi_mean = xr.DataArray(np.random.rand(200,200), dims=('y','x'))

# Build feature stack
fstack = np.stack([ndvi_mean.values.flatten(), ndwi_mean.values.flatten()], axis=1)
mask = ~np.isnan(fstack).any(axis=1)
X = fstack[mask]
# Synthetic labels by thresholding for demo
y = np.zeros(X.shape[0], dtype=int)
y[X[:,0] > 0.6] = 1  # vegetation
y[X[:,1] > 0.5] = 2  # water

clf = RandomForestClassifier(n_estimators=30, random_state=0)
clf.fit(X, y)

pred = np.full(ndvi_mean.shape, np.nan)
pred_flat = np.full(X.shape[0], np.nan)
pred_flat[mask] = clf.predict(X)
pred = pred_flat.reshape(ndvi_mean.shape)

lulc = xr.DataArray(pred, dims=ndvi_mean.dims, coords=ndvi_mean.coords)

plt.figure(figsize=(6,6))
plt.imshow(lulc, cmap='tab10')
plt.title('Demo LULC (RF)')
plt.axis('off')
plt.show()

# Export
lulc.rio.write_crs('EPSG:4326').rio.to_raster('/mnt/data/lulc_ai.tif')
print('Exported /mnt/data/lulc_ai.tif')
